# ImFusion CT Simulation

This notebook demonstrates basic cone-beam projection simulation using ImFusion CT: manual projector-based forward/backward operations and the convenience `ConeBeamSimulation` API.


## Overview
This tutorial shows two ways to simulate cone-beam X-ray projections from a CT volume:
- Low-level projector API for forward/backward operations
- High-level `ConeBeamSimulation` convenience API

### What you will learn
- Preparing a reference projection stack with spacing and geometry
- Applying a GL-based projector for forward/backward ops
- Running a quick simulation

### Prerequisites
- A working `imfusion-sdk` and `imfusion-sdk-computed_tomography` installation with a valid license

## Setup

### Setup python path
This step can be skipped if the package got installed from PyPI.

In [1]:
import sys
import os

# Add the build directory to Python path (adjust if needed)
build_lib_path = '/Users/wieczorek/Desktop/Dev/imfusionsuite/cmake-build-release/lib'
if build_lib_path not in sys.path:
    sys.path.insert(0, build_lib_path)


### Import the modules
We need to import `imfusion`, `imfusion.computed_tomography`, and `imfusion.imagemath.lazy`.

In [2]:
try:
    import imfusion
    import imfusion.computed_tomography as ct
    import imfusion.imagemath.lazy as lazy
except ImportError as e:
    raise ImportError("Failed to import ImFusion CT bindings. Make sure the CT plugin is built and the path is correct.") from e


### Data setup
Unzips demo data if needed and available.

In [3]:
import os, sys
sdk_path = os.path.abspath(os.path.join(os.getcwd(), '../imfusion-sdk'))
if sdk_path not in sys.path:
    sys.path.insert(0, sdk_path)

try:
    from demo_utils import unzip_folder
    zip_path = os.path.join('..', 'imfusion-sdk', 'data', 'pet-ct-rtstruct.zip')
    data_dir = os.path.join('..', 'imfusion-sdk', 'data', 'pet-ct-rtstruct')
    if os.path.exists(zip_path) and not os.path.isdir(data_dir):
        print('Unzipping demo data...')
        unzip_folder(zip_path)
    else:
        print('Demo data is present or archive not found. Skipping unzip.')
except Exception as e:
    print(f"Data setup skipped ({e}).")


Unzipping demo data...


## Steps in this notebook
1) Load a CT volume and center it at the origin
2) Create a synthetic projection stack and configure modern cone-beam geometry
3) Apply projector forward/adjoint with an image-math preprocessing expression
4) Run `ConeBeamSimulation` as a compact alternative

### Load a CT volume and center it at the origin

In [4]:
import numpy as np
volume = imfusion.load("../imfusion-sdk/data/pet-ct-rtstruct/ct.imf")[0]
mat = volume.matrix()
mat[0:3, 3] = [0.0, 0.0, 0.0]
volume.set_matrix(mat)

### Create a synthetic reference projection stack and configure modern cone-beam geometry


In [5]:
reference_projections = imfusion.SharedImageSet()
num_frames = 180
width = height = 1024
for _ in range(num_frames):
    img = imfusion.SharedImage(imfusion.ImageDescriptor(imfusion.PixelType.FLOAT, width, height, 1, 1))
    img.spacing = [400.0 / width, 400.0 / height, 1.0]
    reference_projections.add(img)

ct.make_cone_beam_data(reference_projections)
meta = ct.ConeBeamMetadata.get(reference_projections)
meta.enable_modern_geometry()
param = ct.ParametricGeometryGenerator()
param.source_det_distance = 1200.0
param.source_pat_distance = 800.0
param.angle_range = 180.0
param.transformation_setup.use_iso_center_parameters = True
param.transformation_setup.iso_rotation = [-30.0, 0.0, 45.0]
meta.add_generator(param, select=True)


'ParametricGeometryGenerator-0'

### Apply projector (forward and adjoint) with lazy imagemath expression


In [6]:
projector = ct.GlCBCTProjector(volume, reference_projections)
back_projection = projector.create_domain_sis()
projections = projector.create_range_sis()

# The volume is in Hounsfield units. For the projection we need to shift the values to be positive.

# We could shift the values manually BUT we can also use ImageMath to shift the values directly in the projector.
# This will use lazy evaluation to avoid creating a new memory.
# You can use these expressions also to e.g. mask specific regions of the volume, e.g. metallic objects.

# The following expression will cast the volume to float, truncates the values to the HU range (starting at -1024 for air), and shift the values to be positive.
input_expr = (lazy.astype(lazy.Expression(volume) > -1024, imfusion.PixelType.FLOAT)) * (lazy.Expression(volume) + 1024)
projector.apply(input_expr, volume, projections)
# The same structure applies for the adjoint operator. Let's keep it simple and omit any expressions.
projector.apply_adjoint(projections, back_projection)

imfusion.show([projections, back_projection])

# # Input expression can be a scalar or an expression. E.g. if you want to project 1.0:
# projector.apply(imfusion.imagemath.lazy.Expression(1.0), volume, reference_projections)

# # You can also use output expressions to manipulate the output. (A simple inversion of the values in this example)
# projector.apply(Expression(volume) > -1024 * (Expression(volume) + 1024), volume, reference_projections, -Expression(channel=int(1)))


### Convenience API: ConeBeamSimulation


In [7]:
simulator = ct.ConeBeamSimulation(
    volume[0],
    geometry_preset=ct.GeometryPreset.HALF_SCAN,
    proj_type=ct.ProjectionType.LOG_CONVERTED_ATTENUATION,
    width=1024,
    height=1024,
    frames=360,
    add_poisson_noise=False,
)
projections2 = simulator()

imfusion.show([projections2, volume])
